<a href="https://colab.research.google.com/github/Tanvi-diwakar/AI-and-AI-agents/blob/main/cnn_jellyfish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JellyDrone image classification (CNN) - F21DL Group 11
Run the cells in order on a Colab **T4 GPU** (Runtime > Change runtime type). See `README.md` in this folder for the dataset link and results.
Run step 5 twice: once with `MODEL_TYPE = "resnet18"` and once with `"small_cnn"`.

In [41]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'Jellydrone']


In [42]:
SOURCE = '/content/drive/MyDrive/JellyDrone_Final.v1-jellydrone_raw.yolov8'

print(os.path.exists(SOURCE))

False


In [43]:
import os

path = '/content/drive/MyDrive/Jellydrone'

print("Contents of Jellydrone:")
print(os.listdir(path))

Contents of Jellydrone:
['JellyDrone_Final.v1i.multiclass.zip', 'JellyDrone_Final.v1i.yolov8.zip']


In [44]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Jellydrone/JellyDrone_Final.v1i.multiclass.zip'
extract_path = '/content/jellydrone_local'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")
print(os.listdir(extract_path))

Extraction complete!
['valid', 'README.roboflow.txt', 'train', 'test', 'README.dataset.txt']


In [45]:
for folder in ['train', 'valid', 'test']:
    path = os.path.join(extract_path, folder)
    print(folder, "exists:", os.path.exists(path))
    if os.path.exists(path):
        print(os.listdir(path)[:10])

train exists: True
['AUG27_0055_Chrysaora_0-00-07-71_jpg.rf.549bde499fbb12282c4aee053f80641b.jpg', 'AUG27_0027_Chrysaora_0-00-14-60_jpg.rf.c98f55398c9b0dc62d1cc23c62232637.jpg', 'AUG20_0055_Aurelia_0-00-50-78_jpg.rf.130a3a9b27da342ec57a1ae74835952a.jpg', 'AUG27_0044_Aurelia_0-00-00-83_jpg.rf.8fa10eadace390c734f51d3cab8eaa58.jpg', 'AUG27_0048_Chrysaora_0-00-02-60_jpg.rf.b52873b706b93b4c02b2ed4ccb56f983.jpg', 'AUG20_0040_Aurelia_0-00-04-75_jpg.rf.5a54499598f763de1ed45f4490ee74e3.jpg', 'AUG27_0029_Chrysaora_0-00-19-42_jpg.rf.432feca273132fdccb0a2f14d0cb0e8d.jpg', 'YUN_0019_0-01-06-80_jpg.rf.673957f94e7594a2d0da40beb888f749.jpg', 'frame0-04-27-52_jpg.rf.30042ab5e3b04e9eebca79fa84e010b5.jpg', 'AUG27_0043_Chrysaora_0-00-04-74_jpg.rf.5fa2251e393e72d143990567567c5401.jpg']
valid exists: True
['AUG20_0012_Aurelia_0-00-38-96_jpg.rf.5e210fa1b733fcde55e1ed790fce15a6.jpg', 'AUG27_0005_Chrysaora_0-00-23-04_jpg.rf.0270e2ccf1459d6111e0d472b8090849.jpg', 'AUG20_0056_Aurelia_0-00-01-08_jpg.rf.abcec95710

In [46]:
import os

base = '/content/jellydrone_local'

for split in ['train', 'valid', 'test']:
    folder = os.path.join(base, split)
    files = os.listdir(folder)

    print(f'\n{split}: {len(files)} files')
    print("File types:", {
        ext: sum(f.lower().endswith(ext) for f in files)
        for ext in ['.jpg', '.jpeg', '.png', '.txt', '.xml', '.csv']
    })

print("\nROOT FILES:")
print(os.listdir(base))

print("\nDATASET README:")
with open(os.path.join(base, 'README.dataset.txt')) as f:
    print(f.read()[:3000])


train: 27433 files
File types: {'.jpg': 27432, '.jpeg': 0, '.png': 0, '.txt': 0, '.xml': 0, '.csv': 1}

valid: 2641 files
File types: {'.jpg': 2640, '.jpeg': 0, '.png': 0, '.txt': 0, '.xml': 0, '.csv': 1}

test: 1237 files
File types: {'.jpg': 1236, '.jpeg': 0, '.png': 0, '.txt': 0, '.xml': 0, '.csv': 1}

ROOT FILES:
['valid', 'README.roboflow.txt', 'train', 'test', 'README.dataset.txt']

DATASET README:
# JellyDrone_Final > 2024-02-05 2:34pm
https://universe.roboflow.com/university-of-louisiana-lafayette/jellydrone_final

Provided by a Roboflow user
License: CC BY 4.0




In [47]:
# Inspect the CSV annotation files in train, validation, and test to identify image filenames and their corresponding jellyfish class labels before model training.

import os
import pandas as pd

base = '/content/jellydrone_local'

for split in ['train', 'valid', 'test']:
    folder = os.path.join(base, split)

    csv_file = [f for f in os.listdir(folder) if f.endswith('.csv')][0]

    df = pd.read_csv(os.path.join(folder, csv_file))

    print(f'\n--- {split.upper()} ---')
    print("CSV file:", csv_file)
    print(df.head())
    print("Columns:", df.columns.tolist())


--- TRAIN ---
CSV file: _classes.csv
                                            filename  0  1  2
0  AUG27_0044_Chrysaora_0-00-14-75_jpg.rf.3d841ed...  1  0  1
1  AUG20_0041_Chrysaora_0-02-51-82_jpg.rf.4f7595d...  0  0  0
2  AUG20_0057_Aurelia_0-00-01-32_jpg.rf.9366a9b38...  1  0  0
3  frame0-01-23-83_jpg.rf.86fb3fc0cbc4232fc0fda4e...  0  1  0
4  YUN_0022_0-01-23-72_jpg.rf.47a4eaf268ff43ca1cc...  0  0  0
Columns: ['filename', '0', '1', '2']

--- VALID ---
CSV file: _classes.csv
                                            filename  0  1  2
0  AUG27_0027_Aurelia_0-00-26-01_jpg.rf.5a839617b...  0  0  1
1  AUG27_0043_Chrysaora_0-00-04-37_jpg.rf.c56ba1c...  0  0  0
2  AUG27_0052_Aurelia_0-00-01-63_jpg.rf.1d89e99a1...  0  0  1
3  AUG20_0010_Aurelia_0-02-10-11_jpg.rf.312c067a5...  0  0  0
4  AUG27_0029_Chrysaora_0-00-19-27_jpg.rf.a6175cc...  1  0  0
Columns: ['filename', '0', '1', '2']

--- TEST ---
CSV file: _classes.csv
                                            filename  0  1  2
0  AUG2

In [48]:
# Check the class names and label distribution before converting images into class-wise folders.

import os
import pandas as pd

base = '/content/jellydrone_local'

print(open(os.path.join(base, 'README.roboflow.txt')).read())

for split in ['train', 'valid', 'test']:
    df = pd.read_csv(os.path.join(base, split, '_classes.csv'))
    print(f'\n{split.upper()} label counts:')
    print(df[['0', '1', '2']].sum())


JellyDrone_Final - v1 2024-02-05 2:34pm

This dataset was exported via roboflow.com on November 22, 2026 at 11:39 AM GMT

Roboflow is an end-to-end computer vision platform that helps you
* collaborate with your team on computer vision projects
* collect & organize images
* understand and search unstructured image data
* annotate, and create datasets
* export, train, and deploy computer vision models
* use active learning to improve your dataset over time

For state of the art Computer Vision training notebooks you can use with this dataset,
visit https://github.com/roboflow/notebooks

To find over 100k other datasets and pre-trained models, visit https://universe.roboflow.com

The dataset includes 31308 images.
Jellyfish-Species- are annotated in Multi-Class Classification format.

The following pre-processing was applied to each image:
* Auto-orientation of pixel data (with EXIF-orientation stripping)
* Resize to 640x640 (Stretch)
* Grayscale (CRT phosphor)

The following augmentati

In [49]:
# Verify label combinations and identify jellyfish species names in image filenames before converting the dataset into class-wise folders.

import os
import pandas as pd

base = '/content/jellydrone_local'

for split in ['train', 'valid', 'test']:
    df = pd.read_csv(os.path.join(base, split, '_classes.csv'))

    print(f'\n--- {split.upper()} ---')

    # Count how many classes are assigned to each image
    label_counts = df[['0', '1', '2']].sum(axis=1)
    print("No labels:", (label_counts == 0).sum())
    print("One label:", (label_counts == 1).sum())
    print("Multiple labels:", (label_counts > 1).sum())

    # Check species names appearing in image filenames
    print("\nSpecies names in filenames:")
    print(df['filename'].str.extract(
        r'(Aurelia|Chrysaora|Mastigias)', expand=False
    ).value_counts())


--- TRAIN ---
No labels: 16709
One label: 10006
Multiple labels: 717

Species names in filenames:
filename
Aurelia      13032
Chrysaora     9192
Name: count, dtype: int64

--- VALID ---
No labels: 1598
One label: 989
Multiple labels: 53

Species names in filenames:
filename
Aurelia      1164
Chrysaora     996
Name: count, dtype: int64

--- TEST ---
No labels: 750
One label: 451
Multiple labels: 35

Species names in filenames:
filename
Aurelia      596
Chrysaora    404
Name: count, dtype: int64


In [50]:
# Compare CSV class labels with species names in filenames to identify the correct class mapping.

import os
import pandas as pd

base = '/content/jellydrone_local'

df = pd.read_csv(os.path.join(base, 'train', '_classes.csv'))

# Extract species names from image filenames
df['species'] = df['filename'].str.extract(
    r'(Aurelia|Chrysaora|Mastigias)', expand=False
).fillna('Unknown')

# Compare filename species against CSV labels
print(pd.crosstab(
    df['species'],
    df[['0', '1', '2']].idxmax(axis=1).where(
        df[['0', '1', '2']].sum(axis=1) > 0,
        'No label'
    )
))

# Display examples of images with no labels
print("\nUnlabelled image examples:")
print(df[df[['0', '1', '2']].sum(axis=1) == 0]['filename'].head(10))

col_0         0     1     2  No label
species                              
Aurelia    2082     0  3163      7787
Chrysaora  2406   372  1221      5193
Unknown     234  1245     0      3729

Unlabelled image examples:
1     AUG20_0041_Chrysaora_0-02-51-82_jpg.rf.4f7595d...
4     YUN_0022_0-01-23-72_jpg.rf.47a4eaf268ff43ca1cc...
5     AUG20_0044_Aurelia_0-00-07-56_jpg.rf.0c33b7248...
6     AUG20_0051_Aurelia_0-00-08-94_jpg.rf.018165030...
7     AUG20_0011_Aurelia_0-00-46-25_jpg.rf.93817e5e7...
8     AUG20_0044_Aurelia_0-00-07-57_jpg.rf.87fbe11cd...
9     AUG20_0055_Aurelia_0-00-50-78_jpg.rf.402a36b0c...
10    AUG27_0010_Aurelia_0-00-08-76_jpg.rf.7cb592b71...
12    AUG20_0044_Aurelia_0-00-07-51_jpg.rf.739dbc605...
13    AUG20_0009_Aurelia_0-00-07-27_jpg.rf.6e09a42e6...
Name: filename, dtype: object


In [51]:
# Inspect image filenames for each CSV class to establish the correct jellyfish class mapping.

import os
import pandas as pd

base = '/content/jellydrone_local'
df = pd.read_csv(os.path.join(base, 'train', '_classes.csv'))

for col in ['0', '1', '2']:
    print(f'\nCLASS {col} examples:')
    print(df[df[col] == 1]['filename'].head(10).to_string(index=False))

print('\nMULTIPLE-LABEL EXAMPLES:')
print(df[df[['0', '1', '2']].sum(axis=1) > 1]['filename'].head(10).to_string(index=False))

print('\nUNKNOWN SPECIES EXAMPLES:')
print(df[df['filename'].str.contains('YUN|frame', case=False)]['filename'].head(10).to_string(index=False))


CLASS 0 examples:
AUG27_0044_Chrysaora_0-00-14-75_jpg.rf.3d841edc...
AUG20_0057_Aurelia_0-00-01-32_jpg.rf.9366a9b386...
AUG20_0056_Aurelia_0-00-00-65_jpg.rf.7ad71b7974...
AUG20_0042_Aurelia_0-00-02-02_jpg.rf.051192188d...
AUG20_0042_Aurelia_0-00-01-95_jpg.rf.0152bbdb75...
AUG20_0045_Aurelia_0-00-00-53_jpg.rf.e451c801b0...
AUG20_0009_Aurelia_0-00-06-97_jpg.rf.7e5a5db4a2...
AUG27_0053_Chrysaora_0-00-32-30_jpg.rf.7d097455...
AUG27_0047_Chrysaora_0-00-00-57_jpg.rf.fcd4da7e...
AUG27_0028_Chrysaora_0-02-28-06_jpg.rf.b51e1d43...

CLASS 1 examples:
frame0-01-23-83_jpg.rf.86fb3fc0cbc4232fc0fda4e2...
YUN_0025_0-01-10-69_jpg.rf.c092a2737c7786479a15...
AUG20_0012_Chrysaora_0-00-54-59_jpg.rf.3299b193...
YUN_0028_0-02-03-26_jpg.rf.98185a5252ed56657ad2...
YUN_0028_0-02-03-32_jpg.rf.5331c0e95bdac1021429...
AUG20_0010_Chrysaora_0-01-26-39_jpg.rf.04498969...
AUG20_0010_Chrysaora_0-01-26-64_jpg.rf.1a0e5d06...
AUG20_0010_Chrysaora_0-01-26-75_jpg.rf.e75c776f...
YUN_0025_0-01-10-50_jpg.rf.df27759dcecde0e0e

In [52]:
# Download the new YOLOv8 dataset ZIP from Google Drive without modifying the old dataset.

!pip -q install gdown

!gdown --fuzzy "https://drive.google.com/file/d/19I9J4V_so1x4HI5h1dLFtS0M3Ql5xow0/view?usp=sharing" -O /content/jellydrone_yolov8.zip

Downloading...
From (original): https://drive.google.com/uc?id=19I9J4V_so1x4HI5h1dLFtS0M3Ql5xow0
From (redirected): https://drive.google.com/uc?id=19I9J4V_so1x4HI5h1dLFtS0M3Ql5xow0&confirm=t&uuid=d2549b80-43c9-45c6-92b6-50004f6d2c19
To: /content/jellydrone_yolov8.zip
100% 1.14G/1.14G [00:06<00:00, 168MB/s]


In [53]:
import zipfile

zip_path = '/content/jellydrone_yolov8.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    print('\n'.join(z.namelist()[:30]))

README.dataset.txt
README.roboflow.txt
data.yaml
test/
test/images/
test/images/AUG20_0008_Aurelia0-00-12-66_jpg.rf.58a638aed8b9f06254275c8858b19de9.jpg
test/images/AUG20_0008_Aurelia0-00-12-66_jpg.rf.691f5d54f52607c251a6a45cec23b364.jpg
test/images/AUG20_0008_Aurelia0-00-12-66_jpg.rf.831a524fb08da18f1d0b85e053021e33.jpg
test/images/AUG20_0008_Aurelia0-00-12-66_jpg.rf.903006b179a02a0a7fbb2282f5fa6fa4.jpg
test/images/AUG20_0008_Aurelia0-00-12-71_jpg.rf.322c9b6ebc211b620e58a1f02bd7e304.jpg
test/images/AUG20_0008_Aurelia0-00-12-71_jpg.rf.575de3f985d4ddf9470e0213c8d829ba.jpg
test/images/AUG20_0008_Aurelia0-00-12-71_jpg.rf.866e4480b9e7d0978647a03c4da83a36.jpg
test/images/AUG20_0008_Aurelia0-00-12-71_jpg.rf.ea862c2ae49d682e3cbe589f03cc5f28.jpg
test/images/AUG20_0008_Chrysaora_0-00-45-70_jpg.rf.6100d1c81f5804417d55754e781cfe8a.jpg
test/images/AUG20_0008_Chrysaora_0-00-45-70_jpg.rf.729e3f711fcddfdde6e358312dd31f7f.jpg
test/images/AUG20_0008_Chrysaora_0-00-45-70_jpg.rf.78eb864c1b94540adac1d5e56

In [54]:
# Extract the YOLOv8 dataset into a separate folder and verify its images and annotation files.

import os
import zipfile

zip_path = '/content/jellydrone_yolov8.zip'
extract_path = '/content/jellydrone_yolov8'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")

for split in ['train', 'valid', 'test']:
    split_path = os.path.join(extract_path, split)

    print(f"\n{split.upper()}")
    print(os.listdir(split_path))

    for subfolder in ['images', 'labels']:
        path = os.path.join(split_path, subfolder)
        if os.path.isdir(path):
            print(subfolder, ":", len(os.listdir(path)), "files")

print("\nDATA YAML:")
print(open(os.path.join(extract_path, 'data.yaml')).read())

Extraction complete!

TRAIN
['images', 'labels']
images : 27432 files
labels : 27432 files

VALID
['images', 'labels']
images : 2640 files
labels : 2640 files

TEST
['images', 'labels']
images : 1236 files
labels : 1236 files

DATA YAML:
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['0', '1', '2']

roboflow:
  workspace: university-of-louisiana-lafayette
  project: jellydrone_final
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/university-of-louisiana-lafayette/jellydrone_final/dataset/1


In [55]:

# Convert YOLO bounding-box annotations into class-wise folders using the largest bounding box, count images in each class, and resize them to 256x256 while preserving the original train, valid, and test splits.

import os
import glob
import shutil
from PIL import Image
from collections import Counter

# Dataset paths
source_dir = '/content/jellydrone_yolov8'
output_dir = '/content/jellyfish_preprocessed'

# Image size
IMG_SIZE = (256, 256)

# Original dataset splits
splits = ['train', 'valid', 'test']

# Create output folders
os.makedirs(output_dir, exist_ok=True)

# Process each split separately
for split in splits:

    image_dir = os.path.join(source_dir, split, 'images')
    label_dir = os.path.join(source_dir, split, 'labels')

    image_files = glob.glob(os.path.join(image_dir, '*'))

    class_counts = Counter()
    skipped = 0

    for image_path in image_files:

        filename = os.path.basename(image_path)
        label_path = os.path.join(
            label_dir,
            os.path.splitext(filename)[0] + '.txt'
        )

        # Skip images without annotation files
        if not os.path.exists(label_path):
            skipped += 1
            continue

        # Read YOLO bounding-box annotations
        with open(label_path, 'r') as f:
            annotations = f.readlines()

        boxes = []

        for line in annotations:
            values = line.strip().split()

            if len(values) != 5:
                continue

            class_id = int(float(values[0]))
            width = float(values[3])
            height = float(values[4])

            # Bounding-box area
            area = width * height

            boxes.append((area, class_id))

        # Skip images without valid bounding boxes
        if not boxes:
            skipped += 1
            continue

        # Select the class of the largest bounding box
        largest_box = max(boxes, key=lambda x: x[0])
        class_id = largest_box[1]

        # Create class-wise output folder
        class_dir = os.path.join(
            output_dir, split, f'class_{class_id}'
        )
        os.makedirs(class_dir, exist_ok=True)

        # Resize image to 256x256
        with Image.open(image_path) as img:
            img = img.convert('RGB')
            img = img.resize(IMG_SIZE)
            output_path = os.path.join(class_dir, filename)
            img.save(output_path)

        class_counts[class_id] += 1

    # Display image counts for this split
    print(f'\n{split.upper()} RESULTS')

    for class_id in sorted(class_counts):
        print(f'Class {class_id}: {class_counts[class_id]} images')

    print('Total processed:', sum(class_counts.values()))
    print('Skipped:', skipped)

print('\nPreprocessing completed!')
print('Output folder:', output_dir)



TRAIN RESULTS
Class 0: 4282 images
Class 1: 1617 images
Class 2: 4824 images
Total processed: 10723
Skipped: 16709

VALID RESULTS
Class 0: 399 images
Class 1: 171 images
Class 2: 472 images
Total processed: 1042
Skipped: 1598

TEST RESULTS
Class 0: 175 images
Class 1: 67 images
Class 2: 244 images
Total processed: 486
Skipped: 750

Preprocessing completed!
Output folder: /content/jellyfish_preprocessed


In [56]:
# Verify the number of resized images in each class and dataset split.

import os

base = '/content/jellyfish_preprocessed'

for split in ['train', 'valid', 'test']:
    print(f'\n{split.upper()}')

    split_path = os.path.join(base, split)

    for class_name in sorted(os.listdir(split_path)):
        class_path = os.path.join(split_path, class_name)
        count = len(os.listdir(class_path))
        print(f'{class_name}: {count} images')


TRAIN
class_0: 4282 images
class_1: 1617 images
class_2: 4824 images

VALID
class_0: 399 images
class_1: 171 images
class_2: 472 images

TEST
class_0: 175 images
class_1: 67 images
class_2: 244 images


In [57]:
# Verify that all preprocessed images have been resized to 256x256 pixels.

import os
from PIL import Image

base = '/content/jellyfish_preprocessed'

wrong_size = []

for root, dirs, files in os.walk(base):
    for file in files:
        path = os.path.join(root, file)

        with Image.open(path) as img:
            if img.size != (256, 256):
                wrong_size.append((file, img.size))

print("Images with incorrect dimensions:", len(wrong_size))
print("Expected dimensions: 256x256")
print("Verification complete!")

Images with incorrect dimensions: 0
Expected dimensions: 256x256
Verification complete!
